# E-Commerce Sales & Delivery Analysis

**Business question:** Which categories/states drive revenue, and does delivery speed affect review scores?

Dataset: [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Before running: download the CSVs per `data/README.md` into `data/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

orders = pd.read_csv('data/olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
items = pd.read_csv('data/olist_order_items_dataset.csv')
products = pd.read_csv('data/olist_products_dataset.csv')
customers = pd.read_csv('data/olist_customers_dataset.csv')
reviews = pd.read_csv('data/olist_order_reviews_dataset.csv')
translation = pd.read_csv('data/product_category_name_translation.csv')

orders.shape, items.shape

## 1. Build one analysis table

In [ ]:
df = (
    items
    .merge(orders, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(translation, on='product_category_name', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
)
df = df[df['order_status'] == 'delivered'].copy()
df.shape

## 2. Revenue trend & top categories/states

In [ ]:
monthly_revenue = df.set_index('order_purchase_timestamp')['price'].resample('MS').sum()
monthly_revenue.plot(figsize=(10, 4), title='Monthly Revenue')
plt.ylabel('Revenue (BRL)')
plt.savefig('images/monthly_revenue.png', bbox_inches='tight')
plt.show()

In [ ]:
top_categories = df.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False).head(10)
top_categories.plot(kind='barh', figsize=(8, 6), title='Top 10 Categories by Revenue')
plt.gca().invert_yaxis()
plt.savefig('images/top_categories.png', bbox_inches='tight')
plt.show()

In [ ]:
top_states = df.groupby('customer_state')['price'].sum().sort_values(ascending=False).head(10)
top_states.plot(kind='bar', figsize=(8, 4), title='Top 10 States by Revenue')
plt.savefig('images/top_states.png', bbox_inches='tight')
plt.show()

## 3. Delivery time vs. review score

In [ ]:
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

delivery_vs_review = df.dropna(subset=['delivery_days', 'review_score']).groupby('review_score')['delivery_days'].mean()
delivery_vs_review.plot(kind='bar', figsize=(7, 4), title='Avg Delivery Days by Review Score')
plt.ylabel('Avg delivery days')
plt.savefig('images/delivery_vs_review.png', bbox_inches='tight')
plt.show()

corr = df[['delivery_days', 'review_score']].dropna().corr().iloc[0, 1]
print(f"Correlation between delivery days and review score: {corr:.2f}")

## 4. Key findings & recommendations

_Write 3-5 bullet points summarizing the charts above, then 2-3 sentences of business recommendations. Copy the final version into `README.md`._